# Del texto a los números: embeddings y redes recurrentes

**Sesión 5 · IA Intermedio · EstudIA 2026**

Vamos a clasificar tuits en español en tres clases, **negativo**, **neutral** y **positivo**, de tres maneras: con
una bolsa de palabras, con una red recurrente que aprende sus propios vectores de palabras, y con la misma red
partiendo de vectores que alguien más entrenó con miles de millones de palabras.

**El recorrido.**

| Parte | Qué hacemos |
|---|---|
| 1 | Los datos: tuits en español |
| 2 | Del texto a los números |
| 3 | Una bolsa de palabras |
| 4 | Vectores de palabras |
| 5 | Una red recurrente en PyTorch |
| 6 | Entrenarla con vectores al azar |
| 7 | Entrenarla con vectores preentrenados |
| 8 | Comparar |
| 9 | ¿Importa el orden? |

**Antes de empezar.** *Archivo › Guardar una copia en Drive*. No necesitas GPU: todo el notebook corre en CPU en
pocos minutos. Ejecuta la celda de preparación (▶ o `Shift + Enter`).

In [ ]:
# Preparación: librerías. Ejecútala una vez.
import os, re, time, urllib.request
from collections import Counter
ARCHIVOS = os.environ.get("ESTUDIA_ARCHIVOS", "https://storage.googleapis.com/estud-ia-course-estudia-lab/s05")
TUITS = "https://huggingface.co/api/datasets/cardiffnlp/tweet_sentiment_multilingual/parquet/spanish/{}/0.parquet"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

DISPOSITIVO = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(max(1, os.cpu_count() or 1))
print("Listo.", "Con GPU." if DISPOSITIVO.type == "cuda" else "Sin GPU: no la necesitas para este notebook.")

In [ ]:
# @title Funciones auxiliares: descargar archivos, escribir números y partir textos en palabras
def descargar(nombre, url=None):
    """Baja un archivo (de la carpeta pública del curso, o de `url`) solo la primera vez y devuelve su ruta."""
    ruta = os.path.join("archivos_curso", nombre)
    if not os.path.exists(ruta):
        os.makedirs(os.path.dirname(ruta), exist_ok=True)
        urllib.request.urlretrieve(url or f"{ARCHIVOS}/{nombre}", ruta)
    return ruta

def miles(n):
    """12345 -> "12.345", como escribimos los números en español."""
    return f"{n:,}".replace(",", ".")

def pct(x, decimales=1):
    """0.958 -> "95,8 %"."""
    return f"{100 * x:.{decimales}f} %".replace(".", ",")

def tokens(texto):
    """Parte un texto en palabras y signos, en minúsculas: "No me gustó." -> ["no", "me", "gustó", "."]."""
    return re.findall(r"\w+|[^\w\s]", str(texto).lower())

---
## 1. Los datos: tuits en español

Unos 3.000 tuits en español etiquetados como negativos, neutrales o positivos, con el mismo número de cada clase.
Vienen de un conjunto público para investigación (Barbieri y otros, 2022). Los tuits son reales: algunos tienen
groserías. Aquí solo mostramos ejemplos que revisamos.

In [ ]:
NOMBRES = ["negativo", "neutral", "positivo"]
ent, val, prueba = (pd.read_parquet(descargar(f"tuits_{p}.parquet", TUITS.format(p)))
                    for p in ("train", "validation", "test"))
print(f"Entrenamiento: {miles(len(ent))} · Validación: {miles(len(val))} · Prueba: {miles(len(prueba))}")
print("Tuits por clase en entrenamiento:", {NOMBRES[k]: int(v) for k, v in ent.label.value_counts().sort_index().items()})
for i in [3, 93, 22, 178, 35, 80]:
    print(f"  {NOMBRES[ent.label[i]]:9s} · {ent.text[i]}")

---
## 2. Del texto a los números

Una red no lee letras: lee números. El primer paso es partir el texto en **tokens** (aquí, palabras y signos) y
darle un número a cada palabra del **vocabulario**. Las palabras que no están en el vocabulario comparten un número
especial, el 1. El 0 lo reservamos para rellenar.

In [ ]:
conteo = Counter(w for t in ent.text for w in tokens(t))
VOCAB = {"<relleno>": 0, "<desconocida>": 1}
for palabra, veces in conteo.most_common():
    if veces >= 2:                       # las palabras que aparecen una sola vez no entran
        VOCAB[palabra] = len(VOCAB)
print(f"Vocabulario: {miles(len(VOCAB))} palabras (de {miles(len(conteo))} distintas en el entrenamiento).")

frase = "No me gustó para nada la película, parcero."
print(frase)
print(tokens(frase))
print([VOCAB.get(w, 1) for w in tokens(frase)])

**Para pensar.**

- ¿Qué palabras de la frase quedaron con el número 1? ¿Por qué?
- ¿Qué se pierde al pasar todo a minúsculas? ¿Y qué se gana?

---
## 3. Una bolsa de palabras

La forma más simple de convertir un texto en un vector: contar cuántas veces aparece cada palabra del vocabulario.
Es una **bolsa de palabras**: el vector dice qué palabras hay, no en qué orden. Con esos vectores entrenamos una
regresión logística, como en un problema de tabla.

In [ ]:
bolsa = CountVectorizer(tokenizer=tokens, lowercase=False, token_pattern=None)
X_ent_bolsa = bolsa.fit_transform(ent.text)
clasificador = LogisticRegression(max_iter=3000, C=1.0).fit(X_ent_bolsa, ent.label)
pred_bolsa = clasificador.predict(bolsa.transform(prueba.text))
print(f"Cada tuit es un vector de {miles(X_ent_bolsa.shape[1])} números, casi todos ceros.")
print("Exactitud en prueba de la bolsa de palabras:", pct((pred_bolsa == prueba.label).mean()))

### Dos frases, la misma bolsa

Estas dos frases dicen cosas opuestas, pero tienen exactamente las mismas palabras. Para la bolsa de palabras son
el mismo vector, así que el modelo les da la misma respuesta.

In [ ]:
PARES = [("no fue malo, fue bueno", "no fue bueno, fue malo"),
         ("me gustó, no me aburrí", "me aburrí, no me gustó")]
for a, b in PARES:
    va, vb = bolsa.transform([a]).toarray(), bolsa.transform([b]).toarray()
    pa, pb = clasificador.predict_proba(bolsa.transform([a]))[0], clasificador.predict_proba(bolsa.transform([b]))[0]
    print(f"¿Mismo vector? {bool((va == vb).all())}")
    for texto, p in ((a, pa), (b, pb)):
        print(f"   {texto:28s} → " + " · ".join(f"{NOMBRES[k]} {pct(p[k], 0)}" for k in range(3)))

---
## 4. Vectores de palabras

En la bolsa de palabras, “bueno” y “excelente” son columnas distintas, sin ninguna relación. Un **vector de
palabra** (en inglés, *word embedding*) es una lista de números que ubica cada palabra en un espacio donde las
palabras que aparecen en contextos parecidos quedan cerca.

Usamos los vectores de fastText para el español (Grave y otros, 2018): los aprendió una red leyendo miles de
millones de palabras de internet. Nos quedamos con las 100.000 palabras más frecuentes, 300 números cada una.

In [ ]:
# @title Funciones para los vectores de palabras: vecinos y analogías
_VECTORES = None

def vectores():
    """Las 100.000 palabras más frecuentes del español con su vector de 300 números (fastText). Se baja una vez."""
    global _VECTORES
    if _VECTORES is None:
        datos = np.load(descargar("vectores_es.npz"))
        matriz = datos["vectores"].astype(np.float32)
        matriz /= np.linalg.norm(matriz, axis=1, keepdims=True)
        palabras = datos["palabras"].tolist()
        _VECTORES = (palabras, {p: i for i, p in enumerate(palabras)}, matriz)
    return _VECTORES

def vector_de(palabra):
    """El vector de una palabra, o None si no está entre las 100.000. Prueba también con mayúscula inicial."""
    palabras, indice, matriz = vectores()
    for forma in (palabra, palabra.lower(), palabra.capitalize()):
        if forma in indice:
            return matriz[indice[forma]]
    return None

def vecinos(palabra, cuantos=6):
    """Las palabras con el vector más parecido (similitud del coseno)."""
    palabras, indice, matriz = vectores()
    v = vector_de(palabra)
    if v is None:
        return f"“{palabra}” no está entre las 100.000 palabras."
    orden = np.argsort(-(matriz @ v))
    return [palabras[i] for i in orden[:cuantos + 3] if palabras[i].lower() != palabra.lower()][:cuantos]

def analogia(a, b, c, cuantas=4):
    """b - a + c: “París es a Francia lo que ___ es a Colombia” es analogia("Francia", "París", "Colombia")."""
    palabras, indice, matriz = vectores()
    va, vb, vc = vector_de(a), vector_de(b), vector_de(c)
    q = vb - va + vc
    orden = np.argsort(-(matriz @ (q / np.linalg.norm(q))))
    fuera = {a.lower(), b.lower(), c.lower()}
    return [palabras[i] for i in orden[:20] if palabras[i].lower() not in fuera][:cuantas]

In [ ]:
for palabra in ["Medellín", "arepa", "metro", "excelente"]:
    print(f"{palabra:10s} → {', '.join(vecinos(palabra))}")

Con los vectores también se puede hacer aritmética: “París es a Francia lo que ___ es a Colombia” se calcula como
París − Francia + Colombia.

In [ ]:
print("París - Francia + Colombia =", analogia("Francia", "París", "Colombia"))
print("caminó - caminar + comer   =", analogia("caminar", "caminó", "comer"))

### Lo que también aprendieron

In [ ]:
print("bueno →", vecinos("bueno"))
print("tinto →", vecinos("tinto"))
print("médico - hombre + mujer =", analogia("hombre", "médico", "mujer"))
print("jefe - hombre + mujer   =", analogia("hombre", "jefe", "mujer"))

**Para pensar.**

- ¿Por qué el vecino más cercano de “bueno” es “malo”? ¿Qué problema trae eso para clasificar sentimientos?
- En Colombia, un tinto es un café. ¿Por qué estos vectores lo ponen al lado del vino?
- Los vectores aprendieron de textos escritos por personas. ¿Qué dicen las dos últimas analogías sobre esos textos?

---
## 5. Una red recurrente en PyTorch

Una **red recurrente** lee el texto en orden, una palabra a la vez, y lleva un **estado oculto**: un vector que
resume lo que lleva leído. Con cada palabra nueva, la red combina su vector con el estado anterior y produce el
estado siguiente. Al final, el último estado pasa por una capa que da una probabilidad por clase.

Usamos una **GRU**, una red recurrente con compuertas que deciden qué guardar y qué olvidar, y la hacemos
**bidireccional**: una GRU lee de izquierda a derecha y otra de derecha a izquierda, y juntamos sus dos estados
finales.

In [ ]:
# @title La red recurrente: embedding, GRU y salida
class RedRecurrente(nn.Module):
    """Embedding -> GRU (o LSTM) -> una capa con una salida por clase."""
    def __init__(self, n_palabras, n_clases, dim=128, oculta=128, bidireccional=True, tipo="GRU",
                 pesos=None, congelar=False, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(n_palabras, dim, padding_idx=0)
        if pesos is not None:                      # vectores preentrenados
            self.embedding.weight.data.copy_(torch.as_tensor(pesos))
            self.embedding.weight.requires_grad = not congelar
        capa = nn.GRU if tipo == "GRU" else nn.LSTM
        self.rnn = capa(dim, oculta, batch_first=True, bidirectional=bidireccional)
        self.dropout = nn.Dropout(dropout)
        self.salida = nn.Linear(oculta * (2 if bidireccional else 1), n_clases)

    def forward(self, x):
        largos = (x != 0).sum(1).clamp(min=1).cpu()
        e = self.dropout(self.embedding(x))
        paquete = nn.utils.rnn.pack_padded_sequence(e, largos, batch_first=True, enforce_sorted=False)
        _, h = self.rnn(paquete)
        if isinstance(h, tuple):                   # la LSTM devuelve (h, c)
            h = h[0]
        h = torch.cat(list(h), dim=1)              # el último estado de cada dirección
        return self.salida(self.dropout(h))

In [ ]:
def a_matriz(textos, largo=40):
    """Cada texto se vuelve una fila de números (uno por palabra), rellenada con ceros hasta `largo`."""
    m = np.zeros((len(textos), largo), dtype=np.int64)
    for i, t in enumerate(textos):
        ids = [VOCAB.get(w, 1) for w in tokens(t)][:largo]
        m[i, :len(ids)] = ids
    return torch.from_numpy(m)

X_ent, X_val, X_prueba = a_matriz(ent.text), a_matriz(val.text), a_matriz(prueba.text)
y_ent, y_val, y_prueba = (torch.tensor(d.label.values) for d in (ent, val, prueba))
print("La frase de la parte 2, como fila de la matriz:")
print(a_matriz([frase])[0].tolist())

El ciclo de entrenamiento es el mismo de la sesión 3: predecir, medir el error, calcular gradientes y ajustar.
Al final de cada época medimos la exactitud de validación y guardamos los pesos de la mejor época.

In [ ]:
def entrenar(modelo, epocas=10, tasa=2e-3, lote=32):
    torch.manual_seed(0)
    modelo.to(DISPOSITIVO)
    optimizador = torch.optim.Adam([p for p in modelo.parameters() if p.requires_grad], lr=tasa)
    historia = {"loss": [], "val_loss": [], "val_exactitud": []}
    mejor, mejores_pesos, inicio = -1, None, time.time()
    for epoca in range(epocas):
        modelo.train()
        orden, perdidas = torch.randperm(len(X_ent)), []
        for i in range(0, len(X_ent), lote):
            idx = orden[i:i + lote]
            perdida = F.cross_entropy(modelo(X_ent[idx].to(DISPOSITIVO)), y_ent[idx].to(DISPOSITIVO))
            optimizador.zero_grad(); perdida.backward(); optimizador.step()
            perdidas.append(perdida.item())
        modelo.eval()
        with torch.no_grad():
            salida = modelo(X_val.to(DISPOSITIVO)).cpu()
        exactitud = (salida.argmax(1) == y_val).float().mean().item()
        historia["loss"].append(float(np.mean(perdidas)))
        historia["val_loss"].append(F.cross_entropy(salida, y_val).item())
        historia["val_exactitud"].append(exactitud)
        if exactitud > mejor:
            mejor, mejores_pesos = exactitud, {k: v.clone() for k, v in modelo.state_dict().items()}
    modelo.load_state_dict(mejores_pesos)
    print(f"{epocas} épocas en {time.time() - inicio:.0f} s. Mejor exactitud de validación: {pct(mejor)} "
          f"(época {int(np.argmax(historia['val_exactitud'])) + 1}).")
    return historia

@torch.no_grad()
def predecir(modelo, X):
    modelo.eval()
    return modelo(X.to(DISPOSITIVO)).softmax(1).cpu()

def curvas(historia, titulo):
    epocas = range(1, len(historia["loss"]) + 1)
    fig, ejes = plt.subplots(1, 2, figsize=(11, 3.3))
    ejes[0].plot(epocas, historia["loss"], label="Entrenamiento", color="#000000", lw=2)
    ejes[0].plot(epocas, historia["val_loss"], label="Validación", color="#CB318E", lw=2)
    ejes[0].set_title("Pérdida"); ejes[0].legend(frameon=False)
    ejes[1].plot(epocas, historia["val_exactitud"], color="#CB318E", lw=2); ejes[1].set_title("Exactitud en validación")
    for eje in ejes:
        eje.set_xlabel("Época"); eje.grid(alpha=0.3); eje.spines[["top", "right"]].set_visible(False)
    fig.suptitle(titulo, fontweight="bold"); fig.tight_layout(); plt.show()

---
## 6. Entrenarla con vectores al azar

Primero, la red empieza con vectores de palabras al azar y los aprende junto con todo lo demás, solo con los
1.839 tuits de entrenamiento.

In [ ]:
torch.manual_seed(0)
red_azar = RedRecurrente(len(VOCAB), 3, dim=128)
historia_azar = entrenar(red_azar)
curvas(historia_azar, "GRU bidireccional, vectores al azar")
pred_azar = predecir(red_azar, X_prueba).argmax(1)
print("Exactitud en prueba:", pct((pred_azar == y_prueba).float().mean().item()))

---
## 7. Entrenarla con vectores preentrenados

Ahora la red empieza con los vectores de fastText y los deja **congelados**: solo aprende la GRU y la salida. Es la
misma idea del ajuste fino de la sesión 4, en pequeño: aprovechamos lo que otra red aprendió leyendo muchísimo
texto.

In [ ]:
pesos = np.random.default_rng(0).normal(0, 0.1, (len(VOCAB), 300)).astype(np.float32)
pesos[0] = 0
encontradas = 0
for palabra, i in VOCAB.items():
    v = vector_de(palabra) if i > 1 else None
    if v is not None:
        pesos[i] = v
        encontradas += 1
print(f"{miles(encontradas)} de {miles(len(VOCAB))} palabras del vocabulario tienen vector de fastText.")

torch.manual_seed(0)
red_fasttext = RedRecurrente(len(VOCAB), 3, dim=300, pesos=pesos, congelar=True)
historia_fasttext = entrenar(red_fasttext)
curvas(historia_fasttext, "GRU bidireccional, vectores de fastText congelados")
pred_fasttext = predecir(red_fasttext, X_prueba).argmax(1)
print("Exactitud en prueba:", pct((pred_fasttext == y_prueba).float().mean().item()))

---
## 8. Comparar

In [ ]:
filas = [("Bolsa de palabras", (pred_bolsa == prueba.label.values).mean()),
         ("GRU, vectores al azar", (pred_azar == y_prueba).float().mean().item()),
         ("GRU, vectores de fastText", (pred_fasttext == y_prueba).float().mean().item())]
fig, eje = plt.subplots(figsize=(8, 2.4))
eje.barh(range(3), [f[1] for f in filas], color=["#B0B0B0", "#000000", "#CB318E"])
for i, f in enumerate(filas):
    eje.text(f[1] + 0.01, i, pct(f[1]), va="center")
eje.axvline(1 / 3, color="#7FC200", ls="--", lw=1.5, label="Responder siempre lo mismo (33 %)")
eje.set_yticks(range(3), [f[0] for f in filas]); eje.invert_yaxis(); eje.set_xlim(0, 1)
eje.set_xlabel(f"Exactitud con los {miles(len(prueba))} tuits de prueba"); eje.legend(frameon=False, loc="lower right")
eje.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.show()

ConfusionMatrixDisplay(confusion_matrix(y_prueba, pred_fasttext), display_labels=NOMBRES).plot(cmap="Purples", colorbar=False)
plt.title("GRU con fastText"); plt.xlabel("Lo que respondió"); plt.ylabel("La clase real"); plt.show()

**Para pensar.**

- ¿Por qué las tres formas se quedan cerca del 50 %? ¿Es fácil, para una persona, decir si un tuit es neutral?
- ¿Qué clase confunde más la red? ¿Con cuál?

---
## 9. ¿Importa el orden?

Volvamos a los pares de la parte 3. La bolsa de palabras tenía que darles la misma respuesta. La red recurrente lee
en orden, así que puede responder distinto.

In [ ]:
for a, b in PARES:
    pa, pb = predecir(red_fasttext, a_matriz([a]))[0], predecir(red_fasttext, a_matriz([b]))[0]
    for texto, p in ((a, pa), (b, pb)):
        print(f"   {texto:28s} → " + " · ".join(f"{NOMBRES[k]} {pct(p[k].item(), 0)}" for k in range(3)))
    print()

**Para pensar.**

- ¿Acertó la red en los dos pares? ¿Le basta con leer en orden para entender una negación?
- Si tuvieras que clasificar los mensajes de tu proyecto, ¿empezarías con una bolsa de palabras o con una red recurrente? ¿Por qué?

**Lo que viene.** Las redes recurrentes leen palabra por palabra y tienen que resumir todo en un solo estado. En la
sesión 6 vemos la **atención** y los **transformers**: modelos que miran todas las palabras a la vez y les dan un
vector distinto según el contexto. Volveremos a estos tuits.

**Fuentes.** Tuits: Barbieri, F., Espinosa Anke, L. y Camacho-Collados, J. (2022). XLM-T: Multilingual Language
Models in Twitter for Sentiment Analysis and Beyond. *LREC*. Copia en Hugging Face:
`cardiffnlp/tweet_sentiment_multilingual` (español); licencia CC BY 3.0 y términos de Twitter, para uso académico.
Vectores: Grave, E., Bojanowski, P., Gupta, P., Joulin, A. y Mikolov, T. (2018). Learning Word Vectors for 157
Languages. *LREC*. Licencia CC BY-SA 3.0; nos quedamos con las 100.000 palabras más frecuentes.